# Resume P0 stage 1 FP32 từ checkpoint-3465

- Checkpoint: `alexdannguyen/v2a-stage1-cp3465-1123` (private, full checkpoint ở root dataset).
- Data: `dangthiet/vuln2action-p0-stage1`.
- Resume optimizer/scheduler/RNG tại `global_step=3465` (epoch 5).
- Train thêm 1 epoch (đến epoch 6), giữ effective batch 64: 2xT4, bs 8/GPU, grad accum 4.
- Eval full validation 11.089 dòng, beam 4, FP32.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

data_hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
if not data_hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")
DATA_DIR = Path(data_hits[0]).parent

ckpt_hits = sorted(glob.glob("/kaggle/input/**/trainer_state.json", recursive=True))
ckpt_hits = [p for p in ckpt_hits if "v2a-stage1-cp3465" in p.replace("\\", "/")]
if not ckpt_hits:
    raise FileNotFoundError("checkpoint-3465 trainer_state.json not found")
CKPT = Path(ckpt_hits[0]).parent
assert (CKPT / "model.safetensors").is_file(), CKPT
assert (CKPT / "optimizer.pt").is_file(), CKPT

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("DATA_DIR:", DATA_DIR)
print("CKPT:", CKPT)
print("train:", (DATA_DIR / "flow_train.jsonl").exists())
print("validation:", (DATA_DIR / "flow_validation.jsonl").exists())


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/working/Vuln2Action-Lab")
FULL = Path("/kaggle/working/t5_stage1_resume_cp3465")

cmd = [
    sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
    "scripts/train_t5.py",
    "--dataset", str(DATA_DIR),
    "--output-dir", str(FULL),
    "--model-name", "t5-base",
    "--target-field", "target",
    "--max-input-length", "512",
    "--max-target-length", "512",
    "--per-device-batch-size", "8",
    "--gradient-accumulation-steps", "4",
    "--learning-rate", "5e-5",
    "--num-train-epochs", "50",
    "--save-steps", "693",
    "--eval-steps", "693",
    "--logging-steps", "50",
    "--save-total-limit", "6",
    "--no-load-best-model-at-end",
    "--no-fp16",
    "--gradient-checkpointing",
    "--resume-from-checkpoint", str(CKPT),
    "--stop-after-epoch", "6",
]
print("=== resume training ===")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
import json, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/working/Vuln2Action-Lab")
FULL = Path("/kaggle/working/t5_stage1_resume_cp3465")
EVAL = Path("/kaggle/working/t5_stage1_resume_cp3465_eval_full")

ckpts = sorted(FULL.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if not ckpts:
    raise FileNotFoundError("no checkpoint produced")
EVAL_CKPT = ckpts[-1]
print("=== eval full validation ===")
print("EVAL_CKPT:", EVAL_CKPT)

subprocess.run([
    sys.executable, "scripts/evaluate_t5.py",
    "--checkpoint", str(EVAL_CKPT),
    "--dataset", str(DATA_DIR),
    "--split", "validation",
    "--target-field", "target",
    "--output-dir", str(EVAL),
    "--max-input-length", "512",
    "--max-target-length", "512",
    "--batch-size", "4",
    "--num-beams", "4",
    "--no-fp16",
], cwd=REPO, check=True)

metrics = json.loads((EVAL / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))


In [ ]:
from pathlib import Path
import json

FULL = Path("/kaggle/working/t5_stage1_resume_cp3465")
print("=== training config ===")
cfg_path = FULL / "training_config.json"
if cfg_path.exists():
    print(cfg_path.read_text(encoding="utf-8"))

print("=== final trainer state ===")
state_path = EVAL_CKPT / "trainer_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    keys = ["epoch", "global_step", "best_global_step", "best_metric", "best_model_checkpoint"]
    print(json.dumps({k: state.get(k) for k in keys}, indent=2))
